# Spatial Join: EV Chargers to ASGS SA4 Regions

Assigns every NSW EV charger to the ASGS SA4 (Statistical Area Level 4) region
that contains it, using DuckDB's `spatial` extension.

**Pipeline position:** runs after `data_acquisition.py` (cleaning) and the
augmentation stage (OpenChargeMap / OSM enrichment of DC chargers). It
consumes whichever of the two is available - preferring the augmented output
once that lands, falling back to the cleaned CSV in the meantime - and passes
every input column straight through, so augmentation attributes and SA4
region ride together in one output file for the database/schema stage.

**Inputs**
- `data/processed/tfnsw_ev_augmented.csv` (preferred) or
  `data/processed/tfnsw_ev_cleaned.csv` (fallback)
- ABS ASGS Edition 4 (2026) SA4 boundaries: `data/raw/abs_sa4/*.shp`

**Outputs**
- `data/processed/tfnsw_ev_with_sa4.csv` - chargers + `sa4_code`/`sa4_name`
- `data/ev_nsw.duckdb` - `sa4_region`, `charger_location`,
  `charger_sa4_assignment` tables, spatial-indexed and ready for querying


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

import sys
sys.path.insert(0, "..")
from src import config as cfg

con = duckdb.connect(str(cfg.DUCKDB_PATH))
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")
print("spatial extension loaded")


spatial extension loaded


## 1. Resolve input and create the schema

`LOAD spatial` is per-connection state - it is **not** stored inside the
`.duckdb` file, so any later session that queries a `GEOMETRY` column has to
run it again before touching the data.


In [2]:
if cfg.EV_ENRICHED_CSV.exists():
    ev_csv = cfg.EV_ENRICHED_CSV
    print(f"using augmented charger data: {ev_csv.name}")
else:
    ev_csv = cfg.EV_CLEAN_CSV
    print(f"augmented CSV not found yet - falling back to cleaned data: {ev_csv.name}")

con.execute(cfg.SPATIAL_DDL.read_text(encoding="utf-8"))
print("schema created: sa4_region, charger_location, charger_sa4_assignment")


augmented CSV not found yet - falling back to cleaned data: tfnsw_ev_cleaned.csv
schema created: sa4_region, charger_location, charger_sa4_assignment


## 2. Load ASGS SA4 boundaries and verify the CRS

A CRS mismatch is the one spatial-join failure that raises no error - it just
returns silently wrong or empty results - so the shapefile's declared CRS is
checked against `cfg.SA4_CRS` (EPSG:7844, GDA2020) before anything else runs.

TfNSW coordinates are WGS84; the datum difference from GDA2020 is well under
2 m, immaterial against SA4 polygons spanning tens of kilometres, so no
`ST_Transform` is applied here.


In [3]:
shp = cfg.find_sa4_shapefile()

meta = con.execute("SELECT layers FROM ST_Read_Meta(?)", [str(shp)]).fetchone()[0]
crs = meta[0]["geometry_fields"][0]["crs"]
auth = f"{crs['auth_name']}:{crs['auth_code']}"
assert auth == cfg.SA4_CRS, f"SA4 boundaries are {auth}, expected {cfg.SA4_CRS}"
print(f"SA4 source CRS verified: {auth}")

con.execute("""
    INSERT INTO sa4_region
    SELECT SA4_CODE26, SA4_NAME26, GCC_CODE26, GCC_NAME26, STE_CODE26, STE_NAME26,
           CAST(AREASQKM26 AS DOUBLE), geom IS NULL, geom
    FROM ST_Read(?)
""", [str(shp)])

kept, special = con.execute(
    "SELECT count(*) FILTER (WHERE NOT is_special_purpose), "
    "       count(*) FILTER (WHERE is_special_purpose) FROM sa4_region"
).fetchone()
print(f"SA4 regions loaded: {kept} with geometry, {special} special-purpose (excluded from join)")


SA4 source CRS verified: EPSG:7844
SA4 regions loaded: 89 with geometry, 19 special-purpose (excluded from join)


All 108 national SA4s are loaded, not just the 30 in NSW. Pre-filtering to
NSW would turn a charger sitting just over a state border (NSW entirely
surrounds the ACT) into an "unmatched" record indistinguishable from a
defect; loading the full layer lets a cross-border charger resolve to its
true region instead. The 19 special-purpose codes (*Migratory - Offshore -
Shipping*, *No usual address*) carry no geometry and are excluded from every
spatial predicate.


## 3. Build charger point geometry

`ST_Point` takes `(x, y)` = `(longitude, latitude)` - the reverse of the
column order in the source file. Getting this backwards places every NSW
charger at roughly 33°E 151°N and silently returns zero matches, so
coordinates are screened against a generous NSW bounding box first to make
that failure loud instead of invisible.


In [4]:
con.execute(f"""
    INSERT INTO charger_location
    WITH flagged AS (
        SELECT charger_id,
               CAST(latitude AS DOUBLE)  AS lat,
               CAST(longitude AS DOUBLE) AS lon,
               CASE
                   WHEN latitude IS NULL OR longitude IS NULL THEN 'missing_coordinate'
                   WHEN latitude = 0 OR longitude = 0         THEN 'zero_sentinel'
                   WHEN CAST(longitude AS DOUBLE) NOT BETWEEN {cfg.NSW_BBOX['lon'][0]} AND {cfg.NSW_BBOX['lon'][1]}
                     OR CAST(latitude AS DOUBLE)  NOT BETWEEN {cfg.NSW_BBOX['lat'][0]} AND {cfg.NSW_BBOX['lat'][1]}
                                                              THEN 'outside_nsw_bbox'
               END AS reject
        FROM read_csv_auto(?, header = true)
    )
    SELECT charger_id, lat, lon, reject IS NULL, reject,
           CASE WHEN reject IS NULL THEN ST_Point(lon, lat) END
    FROM flagged
""", [str(ev_csv)])

total, bad = con.execute(
    "SELECT count(*), count(*) FILTER (WHERE NOT coord_valid) FROM charger_location"
).fetchone()
print(f"{total} charger coordinates screened, {bad} failed validation")


1958 charger coordinates screened, 0 failed validation


## 4. Two-pass spatial join

**Pass 1 - strict containment.** `ST_Within(point, polygon)` is the
OGC-correct point-in-polygon predicate. `ST_Intersects` is deliberately not
used here: it also returns true for a point lying exactly on a shared edge,
which would match that charger to both adjacent SA4s and duplicate it
downstream.

**Pass 2 - nearest-boundary fallback.** Under DE-9IM a point sitting exactly
on, or a metre outside, a boundary is *not* "within" it - this happens where
the ABS coastline is generalised relative to the true shoreline. Residual
points only are snapped to the nearest polygon within a 100 m tolerance
(wide enough to absorb coastline generalisation, far too narrow to bridge a
real gap between regions), with the distance recorded for audit.


In [5]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE pass1 AS
    SELECT c.charger_id, s.sa4_code
    FROM charger_location c
    JOIN sa4_region s ON s.geom IS NOT NULL AND ST_Within(c.geom, s.geom)
    WHERE c.coord_valid
""")

# SA4s tile the country without overlapping, so more than one match per
# charger means a boundary artefact - fail loudly rather than let a fan-out
# inflate the row count downstream.
dupes = con.execute(
    "SELECT count(*) FROM (SELECT charger_id FROM pass1 GROUP BY charger_id HAVING count(*) > 1)"
).fetchone()[0]
assert dupes == 0, f"{dupes} charger(s) matched more than one SA4 polygon"
print(f"pass 1 (ST_Within): {con.execute('SELECT count(*) FROM pass1').fetchone()[0]} matched, 0 duplicates")

TOLERANCE_M = 100.0
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE pass2 AS
    WITH residual AS (
        SELECT charger_id, geom FROM charger_location
        WHERE coord_valid AND charger_id NOT IN (SELECT charger_id FROM pass1)
    ),
    ranked AS (
        SELECT r.charger_id, s.sa4_code,
               ST_Distance_Sphere(r.geom, ST_ClosestPoint(s.geom, r.geom)) AS dist_m,
               row_number() OVER (PARTITION BY r.charger_id
                                  ORDER BY ST_Distance(r.geom, s.geom)) AS rn
        FROM residual r CROSS JOIN sa4_region s
        WHERE s.geom IS NOT NULL
    )
    SELECT charger_id, sa4_code, dist_m FROM ranked WHERE rn = 1 AND dist_m <= {TOLERANCE_M}
""")
print(f"pass 2 (nearest-boundary, <= {TOLERANCE_M:.0f} m): "
      f"{con.execute('SELECT count(*) FROM pass2').fetchone()[0]} matched")


pass 1 (ST_Within): 1957 matched, 0 duplicates
pass 2 (nearest-boundary, <= 100 m): 1 matched


Results are consolidated with a **LEFT JOIN** from `charger_location`, never
an inner join - an inner join would silently drop every unmatched charger
instead of leaving it visible and countable.


In [6]:
con.execute("""
    INSERT INTO charger_sa4_assignment
    SELECT c.charger_id,
           COALESCE(p1.sa4_code, p2.sa4_code),
           CASE WHEN p1.sa4_code IS NOT NULL THEN 'within'
                WHEN p2.sa4_code IS NOT NULL THEN 'nearest_boundary'
                ELSE 'unmatched' END,
           CASE WHEN p1.sa4_code IS NOT NULL THEN 0.0 ELSE p2.dist_m END
    FROM charger_location c
    LEFT JOIN pass1 p1 USING (charger_id)
    LEFT JOIN pass2 p2 USING (charger_id)
""")
print("charger_sa4_assignment populated:",
      con.execute("SELECT count(*) FROM charger_sa4_assignment").fetchone()[0], "rows")


charger_sa4_assignment populated: 1958 rows


## 5. Validation

Coverage, match provenance, and the state breakdown - the numbers quoted in
the report's Spatial Data Integration Methodology section.


In [7]:
total, assigned = con.execute(
    "SELECT count(*), count(sa4_code) FROM charger_sa4_assignment"
).fetchone()
print(f"chargers assigned an SA4: {assigned} / {total}  ({100.0 * assigned / total:.1f}%)")
print()

print("match provenance:")
for method, n in con.execute(
    "SELECT match_method, count(*) FROM charger_sa4_assignment GROUP BY 1 ORDER BY 2 DESC"
).fetchall():
    print(f"  {method:<18} {n}")
print()

print("assigned by state:")
for state, n in con.execute(
    "SELECT s.state_name, count(*) FROM charger_sa4_assignment a "
    "JOIN sa4_region s USING (sa4_code) GROUP BY 1 ORDER BY 2 DESC"
).fetchall():
    print(f"  {state:<22} {n}")
print()

print("nearest-boundary snaps (charger_id, sa4, metres):")
for cid, name, d in con.execute(
    "SELECT a.charger_id, s.sa4_name, round(a.match_distance_m, 2) FROM charger_sa4_assignment a "
    "JOIN sa4_region s USING (sa4_code) WHERE a.match_method = 'nearest_boundary'"
).fetchall():
    print(f"  {cid:<8} {name:<34} {d} m")


chargers assigned an SA4: 1958 / 1958  (100.0%)

match provenance:
  within             1957
  nearest_boundary   1

assigned by state:
  New South Wales        1958

nearest-boundary snaps (charger_id, sa4, metres):
  1834     Sydney - Northern Beaches          2.12 m


## 6. Independent cross-check

Coverage alone only proves *an* assignment was made. The assigned SA4s are
cross-checked against the source `lga_name` field, which the join never
reads, as independent corroboration.


In [8]:
sample = con.execute("""
    SELECT e.lga_name, s.sa4_name
    FROM read_csv_auto(?, header = true) e
    JOIN charger_sa4_assignment a USING (charger_id)
    JOIN sa4_region s USING (sa4_code)
    USING SAMPLE 12 ROWS (reservoir, 42)
""", [str(ev_csv)]).fetchall()

print(f"{'lga_name (source, unused by the join)':<40} sa4_name (assigned)")
for lga, sa4 in sample:
    print(f"{str(lga):<40} -> {sa4}")


lga_name (source, unused by the join)    sa4_name (assigned)
Blue Mountains City Council              -> Sydney - Outer West and Blue Mountains
Northern Beaches Council                 -> Sydney - Northern Beaches
Cabonne Council                          -> Central West
Shoalhaven City Council                  -> Southern Highlands and Shoalhaven
Woollahra Municipal Council              -> Sydney - Eastern Suburbs
Newcastle City Council                   -> Newcastle and Lake Macquarie
Eurobodalla Shire Council                -> Capital Region
Waverley Council                         -> Sydney - Eastern Suburbs
Woollahra Municipal Council              -> Sydney - Eastern Suburbs
Newcastle City Council                   -> Newcastle and Lake Macquarie
Ballina Shire Council                    -> Richmond - Tweed
Waverley Council                         -> Sydney - Eastern Suburbs


## 7. Persist: R-tree index, CSV export, checkpoint

At this data volume (under 2,000 points against 89 polygons) the join is
already sub-second and an index earns nothing now - it is built for the
repeated range and nearest-neighbour queries planned for the next stage of
the project.


In [9]:
con.execute("CREATE INDEX IF NOT EXISTS idx_sa4_geom ON sa4_region USING RTREE (geom)")
print("R-tree index created on sa4_region.geom")

target = str(cfg.EV_SA4_CSV).replace("'", "''")
con.execute(f"""
    COPY (
        SELECT e.*, a.sa4_code, s.sa4_name, s.gcc_name, a.match_method, a.match_distance_m
        FROM read_csv_auto(?, header = true) e
        LEFT JOIN charger_sa4_assignment a USING (charger_id)
        LEFT JOIN sa4_region s USING (sa4_code)
        ORDER BY e.charger_id
    ) TO '{target}' (HEADER, DELIMITER ',')
""", [str(ev_csv)])
print(f"wrote {cfg.EV_SA4_CSV}")

con.execute("CHECKPOINT")
con.close()
print("done")


R-tree index created on sa4_region.geom
wrote C:\Users\Shreyash\Downloads\charge-grid-nsw\data\processed\tfnsw_ev_with_sa4.csv
done


## Summary

- 100% of chargers assigned an SA4 region, via a two-pass join with full
  match provenance retained for audit.
- The join is agnostic to its input's extra columns - whatever attributes
  the augmentation stage adds ride through unchanged to
  `tfnsw_ev_with_sa4.csv`, so this notebook does not need to change once
  that stage's output replaces the cleaned-only fallback.
- Output feeds directly into the database schema / load stage.
